In [1]:

import pandas as pd
import os
from pathlib import Path


In [2]:
DATA_DIR = Path(r"../../../data/raw/Yashaswi/HourlyAQI/AQIPollutants/Allhourly/")  # update this

OUTPUT_DIR = DATA_DIR / "by_pollutant"

OUTPUT_DIR.mkdir(exist_ok=True)


## ✅ STEP 1 — Reader + cleaning function (used before append)

In [3]:
def read_and_clean_naps_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        skiprows=7,
        sep=",",
        engine="python",
        on_bad_lines="skip"
    )

    # Clean bilingual headers
    df.columns = [c.split("//")[0].strip() for c in df.columns]

    # Required columns (based on your screenshot)
    required = {"Date", "NAPS ID"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")

    # Type normalization
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.date
    df["NAPS ID"] = df["NAPS ID"].astype(str)

    return df

## ✅ STEP 2 — Build station metadata table (DO THIS ONCE)

In [4]:

def extract_station_metadata(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        skiprows=7,
        sep=",",
        engine="python",
        on_bad_lines="skip"
    )

    df.columns = [c.split("//")[0].strip() for c in df.columns]

    station_table = (
        df[["NAPS ID", "City", "Province/Territory", "Latitude", "Longitude"]]
        .drop_duplicates()
    )

    station_table["NAPS ID"] = station_table["NAPS ID"].astype(str)

    return station_table

## ✅ Create and save station table

In [5]:

first_file = next(DATA_DIR.glob("*.csv"))
station_metadata = extract_station_metadata(first_file)

station_metadata.to_csv("../Yashaswi/AQI_station.csv",
    index=False
)

In [6]:
station_metadata.head()

,NAPS ID,City,Province/Territory,Latitude,Longitude
0,10102,St. John's,NL,47.56038,-52.71147
365,10401,Mount Pearl,NL,47.50513,-52.79476
730,10501,Grand Falls - Windsor,NL,48.92696,-55.65966
1095,10602,Corner Brook,NL,48.95224,-57.92207
1460,30118,Halifax,NS,44.64643,-63.57338


## ✅ STEP 3 — Append data by pollutant (8 separate files)

In [7]:
from collections import defaultdict

pollutant_frames = defaultdict(list)

for file in DATA_DIR.glob("*.csv"):
    pollutant, _ = file.stem.split("_", 1)

    df = read_and_clean_naps_csv(file)
    df["Pollutant"] = pollutant

    pollutant_frames[pollutant].append(df)


In [8]:
for pollutant, frames in pollutant_frames.items():
    combined = pd.concat(frames, ignore_index=True)

    combined = combined.drop_duplicates(
        subset=["Date", "NAPS ID"]
    )

    out_path = OUTPUT_DIR / f"{pollutant}_all_years.csv"
    combined.to_csv(out_path, index=False)

## Check

In [10]:
import pandas as pd

# Final integrity check: one NAPS ID per Date per pollutant file
for f in OUTPUT_DIR.glob("*_all_years.csv"):
    df = pd.read_csv(f, low_memory=False)

    # Normalize types defensively
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.date
    df["NAPS ID"] = df["NAPS ID"].astype(str)

    # Check duplicates
    dupes = df.duplicated(subset=["Date", "NAPS ID"])
    assert not dupes.any(), (
        f"Duplicates found in {f.name}:\n"
        f"{df.loc[dupes, ['Date', 'NAPS ID']].head()}"
    )

print("✅ FINAL CHECK PASSED: No duplicate (Date, NAPS ID) in any pollutant file.")

✅ FINAL CHECK PASSED: No duplicate (Date, NAPS ID) in any pollutant file.


In [16]:
# Example pollutant file
test = pd.read_csv(OUTPUT_DIR /"PM25_all_years.csv")


test["NAPS ID"].dtype        # should be object (string)
test["Pollutant"].unique()  # should be ONE pollutant only

<StringArray>
['PM25']
Length: 1, dtype: str

In [17]:
test.tail(10)

,Pollutant,Method Code,NAPS ID,City,Province/Territory,Latitude,Longitude,Date,H01,H02,...,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24
1196235,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-22,2,3,...,2,2,2,3,4,2,2,2,2,2
1196236,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-23,2,2,...,1,1,2,1,2,2,2,2,2,2
1196237,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-24,1,1,...,1,1,3,3,1,2,2,3,4,1
1196238,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-25,1,1,...,2,1,1,3,3,1,2,1,1,1
1196239,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-26,1,1,...,0,1,1,0,1,1,1,1,1,1
1196240,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-27,1,1,...,3,2,1,1,2,2,3,4,4,5
1196241,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-28,5,4,...,4,4,5,8,7,7,6,4,3,3
1196242,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-29,3,3,...,2,3,3,3,6,3,2,2,3,2
1196243,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-30,4,4,...,2,2,2,2,2,2,2,3,4,5
1196244,PM25,636,119004,Whitehorse,YU,60.71972,-135.05875,2024-12-31,4,2,...,2,2,5,9,22,24,16,8,9,15


## Big file check

In [18]:
# Example pollutant file
O3_test = pd.read_csv(OUTPUT_DIR / "O3_all_years.csv")


O3_test["NAPS ID"].dtype        # should be object (string)
O3_test["Pollutant"].unique()  # should be ONE pollutant only

<StringArray>
['O3']
Length: 1, dtype: str

In [28]:
O3_test.shape

(1229895, 31)

In [19]:
O3_test["Date"] = pd.to_datetime(O3_test["Date"])

In [20]:
date_naps_counts = (
    O3_test
    .groupby("Date")["NAPS ID"]
    .nunique()
)

multiple_naps_same_date = date_naps_counts[date_naps_counts > 1]

In [21]:
multiple_naps_same_date.head()

Date
2010-01-01    193
2010-01-02    193
2010-01-03    193
2010-01-04    193
2010-01-05    193
Name: NAPS ID, dtype: int64

In [22]:
records_multiple_naps = (
    O3_test
    .merge(
        multiple_naps_same_date.rename("n_naps"),
        on="Date",
        how="inner"
    )
    .sort_values("Date")
)

records_multiple_naps.head(5)


,Pollutant,NAPS ID,City,Province/Territory,Latitude,Longitude,Date,H01,H02,H03,...,H16,H17,H18,H19,H20,H21,H22,H23,H24,n_naps
0,O3,10102,St. John's,NL,47.56038,-52.71147,2010-01-01,33,33,33,...,20,23,16,17,18,10,5,14,17,193
47085,O3,70119,Winnipeg,MB,49.89799,-97.14653,2010-01-01,23,22,23,...,22,19,12,9,10,9,7,2,3,193
45260,O3,65801,Chatham,ON,42.40369,-82.20831,2010-01-01,20,24,28,...,34,37,38,39,38,38,36,36,36,193
51830,O3,90302,Red Deer,AB,52.29867,-113.79352,2010-01-01,16,-999,21,...,20,18,16,15,12,8,13,8,11,193
36135,O3,60903,London,ON,43.00672,-81.20642,2010-01-01,13,19,22,...,29,29,29,30,32,34,36,36,36,193


In [23]:
(O3_test.groupby("Date")["NAPS ID"].nunique() > 1).any()

np.True_

In [44]:

records_multiple_naps.groupby("Date").agg(
    naps_ids=("NAPS ID", lambda x: sorted(x.unique())),
    count=("NAPS ID", "nunique")
)


,naps_ids,count
Date,,
2010-01-01,"[10102, 10401, 10501, 10601, 10602, 10801, 301...",193
2010-01-02,"[10102, 10401, 10501, 10601, 10602, 10801, 301...",193
2010-01-03,"[10102, 10401, 10501, 10601, 10602, 10801, 301...",193
2010-01-04,"[10102, 10401, 10501, 10601, 10602, 10801, 301...",193
2010-01-05,"[10102, 10401, 10501, 10601, 10602, 10801, 301...",193
...,...,...
2024-12-27,"[10102, 10401, 10501, 10601, 10602, 10801, 109...",238
2024-12-28,"[10102, 10401, 10501, 10601, 10602, 10801, 109...",238
2024-12-29,"[10102, 10401, 10501, 10601, 10602, 10801, 109...",238


In [24]:

O3_test.groupby(["Date", "NAPS ID"]).size().reset_index(name="n").query("n > 1")


,Date,NAPS ID,n


In [25]:
dupes = O3_test[O3_test.duplicated(
    subset=["Date", "NAPS ID"], keep=False
)].sort_values(["Date", "NAPS ID"])

In [26]:
dupes.groupby(["Date", "NAPS ID"]).nunique()

,,Pollutant,City,Province/Territory,Latitude,Longitude,H01,H02,H03,H04,H05,...,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24
Date,NAPS ID,,,,,,,,,,,,,,,,,,,,,


In [51]:
dupes.groupby(["Date", "NAPS ID"]).nunique()


Pollutant  H01  H02  H03  H04  H05  H06  H07  H08  H09  \
Date       NAPS ID                                                           
2010-01-01 10102            1    1    1    1    1    1    1    1    1    1   
           10401            1    1    1    1    1    1    1    1    1    1   
           10501            1    1    1    1    1    1    1    1    1    1   
           10601            1    1    1    1    1    1    1    1    1    1   
           10602            1    1    1    1    1    1    1    1    1    1   
...                       ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
2024-12-31 129203           1    1    1    1    1    1    1    1    1    1   
           129401           1    1    1    1    1    1    1    1    1    1   
           129501           1    1    1    1    1    1    1    1    1    1   
           129602           1    1    1    1    1    1    1    1    1    1   
           129701           1    1    1    1    1    1    1    1    1    1   

                    ...  H15  H16  H17  H18  H19  H20  H21  H22  H23  H24  
Date       NAPS ID  ...                                                    
2010-01-01 10102    ...    1    1    1    1    1    1    1    1    1    1  
           10401    ...    1    1    1    1    1    1    1    1    1    1  
           10501    ...    1    1    1    1    1    1    1    1    1    1  
           10601    ...    1    1    1    1    1    1    1    1    1    1  
           10602    ...    1    1    1    1    1    1    1    1    1    1  
...                 ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  
2024-12-31 129203   ...    1    1    1    1    1    1    1    1    1    1  
           129401   ...    1    1    1    1    1    1    1    1    1    1  
           129501   ...    1    1    1    1    1    1    1    1    1    1  
           129602   ...    1    1    1    1    1    1    1    1    1    1  
           129701   ...    1    1    1    1    1    1    1    1    1    1  

[1229895 rows x 25 columns]

In [52]:
O3_test.shape[0], O3_test.drop_duplicates().shape[0]

(2459790, 1229895)

In [53]:
O3_test = O3_test.drop_duplicates().reset_index(drop=True)